<a href="https://colab.research.google.com/github/annisafadila/1320123001-data-mining/blob/main/mental_health_pwa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🚀 PHASE 1 — SETUP AWAL (WAJIB)
1. Install yang dibutuhkan
Node.js
MySQL / XAMPP
Sublime Text

2. Buat folder project

mental-health-pwa/
│
├── frontend/
├── backend/

🧱 *PHASE 2 — BACKEND (OTAK SISTEM)*
3. Setup backend

Masuk folder backend:

In [ ]:
cd backend
npm init -y
npm install express mysql2 cors body-parser axios jsonwebtoken bcryptjs

Buat database (MySQL)

In [ ]:
CREATE DATABASE mental_health;
USE mental_health;

CREATE TABLE users (
    id INT AUTO_INCREMENT PRIMARY KEY,
    username VARCHAR(50),
    password VARCHAR(255)
);

CREATE TABLE chats (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    message TEXT,
    reply TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE moods (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT,
    mood INT,
    created_at DATE
);

5. Buat koneksi database (db.js)

In [ ]:
const mysql = require("mysql2");

const db = mysql.createConnection({
    host: "localhost",
    user: "root",
    password: "",
    database: "mental_health"
});

module.exports = db;

6. Buat server (server.js)
🔐 Setup + JWT

In [ ]:
const express = require("express");
const bodyParser = require("body-parser");
const cors = require("cors");
const axios = require("axios");
const jwt = require("jsonwebtoken");
const bcrypt = require("bcryptjs");
const db = require("./db");

const app = express();
app.use(cors());
app.use(bodyParser.json());

const SECRET = "rahasia";

🔐 REGISTER

In [ ]:
app.post("/register", async (req,res)=>{
    const {username,password} = req.body;
    const hash = await bcrypt.hash(password,10);

    db.query("INSERT INTO users (username,password) VALUES (?,?)",
    [username,hash]);

    res.json({message:"register berhasil"});
});

🔐 LOGIN JWT

In [ ]:
app.post("/login",(req,res)=>{
    const {username,password} = req.body;

    db.query("SELECT * FROM users WHERE username=?",
    [username],
    async (err,result)=>{
        if(result.length===0) return res.json({success:false});

        const user = result[0];
        const match = await bcrypt.compare(password,user.password);

        if(!match) return res.json({success:false});

        const token = jwt.sign({id:user.id},SECRET);
        res.json({success:true,token});
    });
});

🔒 Middleware Token

In [ ]:
function verifyToken(req,res,next){
    const token = req.headers.authorization;
    if(!token) return res.sendStatus(403);

    jwt.verify(token,SECRET,(err,decoded)=>{
        if(err) return res.sendStatus(401);
        req.userId = decoded.id;
        next();
    });
}

🤖 CHATBOT EMPATIK

In [ ]:
app.post("/chat", verifyToken, async (req,res)=>{
    const {message} = req.body;

    const response = await axios.post(
        "https://api.openai.com/v1/chat/completions",
        {
            model:"gpt-4.1-mini",
            messages:[
                {
                    role:"system",
                    content:"Kamu adalah konselor yang empatik, lembut, dan membantu mahasiswa yang stres."
                },
                {role:"user", content: message}
            ]
        },
        {
            headers:{
                "Authorization":"Bearer YOUR_API_KEY"
            }
        }
    );

    const reply = response.data.choices[0].message.content;

    db.query("INSERT INTO chats (user_id,message,reply) VALUES (?,?,?)",
    [req.userId,message,reply]);

    res.json({reply});
});

😊 MOOD

In [ ]:
app.post("/mood", verifyToken,(req,res)=>{
    const {mood} = req.body;

    db.query("INSERT INTO moods (user_id,mood,created_at) VALUES (?,?,CURDATE())",
    [req.userId,mood]);

    res.json({message:"ok"});
});

app.get("/mood", verifyToken,(req,res)=>{
    db.query("SELECT * FROM moods WHERE user_id=?",
    [req.userId],
    (err,result)=> res.json(result));
});

🧾 HISTORY

In [ ]:
app.get("/history", verifyToken,(req,res)=>{
    db.query("SELECT * FROM chats WHERE user_id=? ORDER BY id DESC",
    [req.userId],
    (err,result)=> res.json(result));
});

▶️ Jalankan server

In [ ]:
node server.js

🎨 PHASE 3 — FRONTEND (TAMPILAN)
7. login.html

In [ ]:
<div class="container">
  <h2>Login</h2>
  <input id="username" placeholder="Username">
  <input id="password" type="password">
  <button onclick="login()">Login</button>
</div>

8. dashboard.html

In [ ]:
<div class="container">

<h3>Mood Hari Ini</h3>
<select id="mood">
  <option value="1">😢</option>
  <option value="2">😐</option>
  <option value="3">😊</option>
</select>
<button onclick="saveMood()">Simpan</button>

<canvas id="chart"></canvas>

<h3>Curhat</h3>
<textarea id="msg"></textarea>
<button onclick="send()">Kirim</button>

<div id="reply"></div>

<h3>Riwayat</h3>
<div id="history"></div>

</div>

9. style.css (mobile fresh)

body{
  font-family: 'Segoe UI';
  background:#E8F5E9;
  margin:0;
}

.container{
  max-width:400px;
  margin:auto;
  padding:15px;
}

button{
  width:100%;
  padding:12px;
  background:#4CAF50;
  color:white;
  border:none;
  border-radius:15px;
}

10. app.js (inti frontend)
🔐 LOGIN

In [ ]:
async function login(){
    const res = await fetch("http://localhost:3000/login",{
        method:"POST",
        headers:{"Content-Type":"application/json"},
        body:JSON.stringify({
            username:username.value,
            password:password.value
        })
    });

    const data = await res.json();

    if(data.success){
        localStorage.setItem("token",data.token);
        location.href="dashboard.html";
    }
}

🤖 CHAT

In [ ]:
async function send(){
    const token = localStorage.getItem("token");

    const res = await fetch("http://localhost:3000/chat",{
        method:"POST",
        headers:{
            "Content-Type":"application/json",
            "Authorization":token
        },
        body:JSON.stringify({message:msg.value})
    });

    const data = await res.json();
    reply.innerText = data.reply;
}

😊 MOOD + 📊 CHART

In [ ]:
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>

In [ ]:
async function loadChart(){
    const token = localStorage.getItem("token");

    const res = await fetch("http://localhost:3000/mood",{
        headers:{Authorization:token}
    });

    const data = await res.json();

    new Chart(chart,{
        type:'line',
        data:{
            labels:data.map(d=>d.created_at),
            datasets:[{data:data.map(d=>d.mood)}]
        }
    });
}

🧾 HISTORY



In [ ]:
async function loadHistory(){
    const token = localStorage.getItem("token");

    const res = await fetch("http://localhost:3000/history",{
        headers:{Authorization:token}
    });

    const data = await res.json();

    history.innerHTML = data.map(d=>
      `<p>${d.message}</p><p>${d.reply}</p>`
    ).join("");
}

📱 PHASE 4 — PWA
manifest.json

In [ ]:
{
  "name":"Mental App",
  "short_name":"Mental",
  "start_url":"login.html",
  "display":"standalone",
  "theme_color":"#4CAF50"
}

service-worker.js

In [ ]:
self.addEventListener("install", e=>{
  e.waitUntil(
    caches.open("app").then(cache=>{
      return cache.addAll(["/","/login.html"]);
    })
  );
});

🎯 URUTAN PENGERJAAN (PALING BENAR)
Buat folder
Setup backend
Buat database
Buat login (WAJIB berhasil dulu)
Tambah JWT
Dashboard
Chatbot
Mood
Grafik
Riwayat
PWA